In [15]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [1]:
OBJECTIVE = """
Find the cheapest model-routing strategy that satisfies
our minimum quality and latency requirements.
"""

# Hard constraints
MIN_QUALITY = 0.95
MAX_QUALITY_DROP = 0.02
MAX_LATENCY_MS = 1500

# Cost baseline
BASELINE_STRATEGY = "frontier_only"

# Strategies we will experimentally compare
ROUTING_STRATEGIES = [
    "frontier_only",
    "difficulty_based",
    "quality_aware",
    "quality_aware_with_escalation",
]

# Primary evaluation metrics
METRICS = [
    "total_cost",
    "cost_reduction_pct",
    "avg_quality",
    "quality_delta",
    "p95_latency_ms",
    "cache_hit_rate",
    "escalation_rate",
]

print("Objective:")
print(OBJECTIVE)

print("\nConstraints:")
print(f"Minimum quality       : {MIN_QUALITY}")
print(f"Maximum quality drop  : {MAX_QUALITY_DROP}")
print(f"Maximum latency       : {MAX_LATENCY_MS} ms")
print(f"Baseline              : {BASELINE_STRATEGY}")

print("\nStrategies:")
for strategy in ROUTING_STRATEGIES:
    print(f"  - {strategy}")

Objective:

Find the cheapest model-routing strategy that satisfies
our minimum quality and latency requirements.


Constraints:
Minimum quality       : 0.95
Maximum quality drop  : 0.02
Maximum latency       : 1500 ms
Baseline              : frontier_only

Strategies:
  - frontier_only
  - difficulty_based
  - quality_aware
  - quality_aware_with_escalation


In [3]:
from typing import Any, Optional

from pydantic import BaseModel, Field, ConfigDict


class LLMRequest(BaseModel):
    """
    Raw request entering our optimizer.

    Important:
    - These are caller-provided facts.
    - No routing decision belongs here.
    - No predicted difficulty belongs here.
    """

    model_config = ConfigDict(extra="forbid")

    request_id: str = Field(
        ...,
        description="Unique identifier for tracing and evaluation."
    )

    user_prompt: str = Field(
        ...,
        min_length=1,
        description="The user's actual request."
    )

    system_prompt: Optional[str] = Field(
        default=None,
        description="Optional system instruction."
    )

    context: Optional[str] = Field(
        default=None,
        description="Optional retrieved or application-provided context."
    )

    max_output_tokens: int = Field(
        default=512,
        gt=0,
        le=32_000,
        description="Maximum expected completion length."
    )

    metadata: dict[str, Any] = Field(
        default_factory=dict,
        description="Application metadata; not trusted for routing decisions."
    )

In [7]:
request = LLMRequest(
    request_id="req_001",
    user_prompt="Explain why database indexes improve query performance.",
    system_prompt="You are a helpful technical assistant.",
    context="The user is preparing for a DBMS interview.",
    max_output_tokens=300
)

request

LLMRequest(request_id='req_001', user_prompt='Explain why database indexes improve query performance.', system_prompt='You are a helpful technical assistant.', context='The user is preparing for a DBMS interview.', max_output_tokens=300, metadata={})

In [11]:
from typing import Literal

from pydantic import BaseModel, AnyHttpUrl, ConfigDict


class ProviderDefinition(BaseModel):
    """
    Static definition owned by our application.

    The user never supplies the base URL.
    """

    model_config = ConfigDict(frozen=True)

    provider_id: str
    display_name: str
    base_url: AnyHttpUrl
    protocol: Literal["openai_compatible"] = "openai_compatible"


PROVIDERS = {
    "openai": ProviderDefinition(
        provider_id="openai",
        display_name="OpenAI",
        base_url="https://api.openai.com/v1",
    ),

    "groq": ProviderDefinition(
        provider_id="groq",
        display_name="Groq",
        base_url="https://api.groq.com/openai/v1",
    ),

    "together": ProviderDefinition(
        provider_id="together",
        display_name="Together AI",
        base_url="https://api.together.xyz/v1",
    ),

    "openrouter": ProviderDefinition(
        provider_id="openrouter",
        display_name="OpenRouter",
        base_url="https://openrouter.ai/api/v1",
    ),

    "deepseek": ProviderDefinition(
        provider_id="deepseek",
        display_name="DeepSeek",
        base_url="https://api.deepseek.com/v1",
    ),

    "baseten": ProviderDefinition(
        provider_id="baseten",
        display_name="Baseten",
        base_url="https://inference.baseten.co/v1",
    ),
}


print(f"Registered providers: {len(PROVIDERS)}")

for provider_id, provider in PROVIDERS.items():
    print(
        f"{provider_id:12} → "
        f"{provider.display_name:15} → "
        f"{provider.base_url}"
    )

Registered providers: 6
openai       → OpenAI          → https://api.openai.com/v1
groq         → Groq            → https://api.groq.com/openai/v1
together     → Together AI     → https://api.together.xyz/v1
openrouter   → OpenRouter      → https://openrouter.ai/api/v1
deepseek     → DeepSeek        → https://api.deepseek.com/v1
baseten      → Baseten         → https://inference.baseten.co/v1


In [16]:
from openai import OpenAI

provider = PROVIDERS["baseten"]

client = OpenAI(
    api_key=os.getenv("BASETEN_API_KEY"),
    base_url=str(provider.base_url),
)

### User Provider Connection

In [17]:
from pydantic import SecretStr


class ProviderConnection(BaseModel):
    """
    Runtime connection supplied by the user.

    The user provides:
    - provider_id
    - API key

    The base URL comes from our Provider Catalog.
    """

    provider_id: str = Field(
        ...,
        description="ID of a provider from our provider catalog."
    )

    api_key: SecretStr = Field(
        ...,
        min_length=1,
        description="User's provider API key."
    )

In [18]:
connection = ProviderConnection(
    provider_id="openai",
    api_key="sk-example-secret-key",
)

print(connection)

provider_id='openai' api_key=SecretStr('**********')


In [19]:
print(connection.api_key.get_secret_value())

sk-example-secret-key


### Provider Registry

In [21]:
class ProviderRegistry:
    """
    Provides controlled access to the application's
    supported provider catalog.
    """

    def __init__(self, providers: dict[str, ProviderDefinition]):
        self._providers = providers

    def get(self, provider_id: str) -> ProviderDefinition:
        """
        Return the provider definition.

        Raises:
            ValueError: if the provider is unsupported.
        """
        try:
            return self._providers[provider_id]
        except KeyError:
            raise ValueError(
                f"Unsupported provider: '{provider_id}'"
            )

    def exists(self, provider_id: str) -> bool:
        """Check whether a provider is supported."""
        return provider_id in self._providers

    def list(self) -> list[ProviderDefinition]:
        """Return all registered providers."""
        return list(self._providers.values())


provider_registry = ProviderRegistry(PROVIDERS)

In [22]:
provider = provider_registry.get("openai")

print(provider)

provider_id='openai' display_name='OpenAI' base_url=AnyHttpUrl('https://api.openai.com/v1') protocol='openai_compatible'


In [23]:
print(provider.display_name)
print(provider.base_url)

OpenAI
https://api.openai.com/v1


In [25]:
print(provider_registry.exists("openai"))
print(provider_registry.exists("does_not_exist"))

True
False


In [26]:
for provider in provider_registry.list():
    print(
        provider.provider_id,
        "→",
        provider.display_name
    )

openai → OpenAI
groq → Groq
together → Together AI
openrouter → OpenRouter
deepseek → DeepSeek
baseten → Baseten


### OpenAI-Compatible Client Factory

In [27]:
from openai import OpenAI


class OpenAIClientFactory:
    """
    Creates OpenAI-compatible clients from a user connection.

    Provider-specific base URLs come from our ProviderRegistry.
    The API key comes from the user's ProviderConnection.
    """

    def __init__(self, provider_registry: ProviderRegistry):
        self.provider_registry = provider_registry

    def create(self, connection: ProviderConnection) -> OpenAI:
        provider = self.provider_registry.get(
            connection.provider_id
        )

        return OpenAI(
            api_key=connection.api_key.get_secret_value(),
            base_url=str(provider.base_url),
        )


client_factory = OpenAIClientFactory(provider_registry)

In [28]:
# Test — Client Construction
connection = ProviderConnection(
    provider_id="openai",
    api_key="sk-example-key",
)

client = client_factory.create(connection)

print(type(client))

<class 'openai.OpenAI'>


In [30]:
connection = ProviderConnection(
    provider_id="groq",
    api_key="gsk-example-key",
)

client = client_factory.create(connection)

print(client.base_url)

https://api.groq.com/openai/v1/


### API Key Setup

In [31]:
import os
from getpass import getpass


BASETEN_API_KEY = os.getenv("BASETEN_API_KEY")

if not BASETEN_API_KEY:
    BASETEN_API_KEY = getpass("Enter BaseTen API key: ")

print("API key loaded:", bool(BASETEN_API_KEY))

API key loaded: True


In [40]:
openai_connection = ProviderConnection(
    provider_id="baseten",
    api_key=BASETEN_API_KEY,
)

In [41]:
client = client_factory.create(openai_connection)
client.models.list()

SyncPage[Model](data=[Model(id='openai/gpt-oss-120b', created=1754410981, object='model', owned_by='baseten', shutdown_date=None, name='OpenAI GPT 120B', description='Extremely capable general-purpose LLM with strong, controllable reasoning capabilities', context_length=128072, max_completion_tokens=128072, quantization='fp4', pricing={'prompt': '0.0000001', 'completion': '0.0000005', 'image': '0', 'request': '0', 'input_cache_read': '0.0000001'}, supported_sampling_parameters=['temperature', 'stop'], supported_features=['tools', 'reasoning', 'json_mode', 'structured_outputs', 'reasoning_effort'], input_modalities=['text'], output_modalities=['text']), Model(id='zai-org/GLM-4.7', created=1766509343, object='model', owned_by='baseten', shutdown_date=None, name='GLM 4.7', description='Fast general-purpose LLM with extended 200K context window, superior coding and reasoning capabilities and enhanced tool use for agentic workflows.', context_length=200000, max_completion_tokens=200000, qua

### Model Discovery

In [38]:
class ModelDiscoveryService:
    """
    Discovers models exposed by an OpenAI-compatible provider.
    """

    def __init__(self, client_factory: OpenAIClientFactory):
        self.client_factory = client_factory

    def discover(
        self,
        connection: ProviderConnection,
    ):
        client = self.client_factory.create(connection)

        response = client.models.list()

        return response.data


model_discovery = ModelDiscoveryService(client_factory)

In [39]:
models = model_discovery.discover(openai_connection)

print(f"Discovered models: {len(models)}")

Discovered models: 16


In [42]:
for model in models[:10]:
    print(model.id)

openai/gpt-oss-120b
zai-org/GLM-4.7
moonshotai/Kimi-K2.6
deepseek-ai/DeepSeek-V4-Pro
nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B
zai-org/GLM-5.2
moonshotai/Kimi-K2.7-Code
deepseek-ai/DeepSeek-V4-Flash-0731
thinkingmachines/inkling
zai-org/GLM-5.2-Fast


### Normalized Model Profile

In [46]:
from typing import Any


class ModelPricing(BaseModel):
    prompt: float | None = None
    completion: float | None = None
    input_cache_read: float | None = None
    image: float | None = None
    request: float | None = None


class ModelProfile(BaseModel):
    """
    Normalized provider model information.

    This represents facts discovered from the provider.
    It does NOT contain our routing decisions.
    """

    model_config = ConfigDict(frozen=True)

    provider_id: str

    model_id: str
    display_name: str

    description: str | None = None

    context_length: int | None = None
    max_completion_tokens: int | None = None

    pricing: ModelPricing | None = None

    supported_features: tuple[str, ...] = ()
    supported_sampling_parameters: tuple[str, ...] = ()

    input_modalities: tuple[str, ...] = ()
    output_modalities: tuple[str, ...] = ()

### Model Normalizer

In [64]:
class ModelNormalizer:

    @staticmethod
    def normalize(
        provider_id: str,
        model: Any,
    ) -> ModelProfile:

        pricing_data = getattr(model, "pricing", None) or {}

        # pricing = ModelPricing(
        #     prompt=float(pricing_data["prompt"])
        #     if pricing_data.get("prompt") is not None
        #     else None,

        #     completion=float(pricing_data["completion"])
        #     if pricing_data.get("completion") is not None
        #     else None,

        #     input_cache_read=float(pricing_data["input_cache_read"])
        #     if pricing_data.get("input_cache_read") is not None
        #     else None,

        #     image=float(pricing_data["image"])
        #     if pricing_data.get("image") is not None
        #     else None,

        #     request=float(pricing_data["request"])
        #     if pricing_data.get("request") is not None
        #     else None,
        # )

        pricing_data = getattr(model, "pricing", None) or {}

        pricing = ModelPricing(
            prompt=parse_token_price(
                pricing_data.get("prompt")
            ),

            completion=parse_token_price(
                pricing_data.get("completion")
            ),

            input_cache_read=parse_token_price(
                pricing_data.get("input_cache_read")
            ),

            image=parse_token_price(
                pricing_data.get("image")
            ),

            request=parse_token_price(
                pricing_data.get("request")
            ),
        )

        return ModelProfile(
            provider_id=provider_id,
            model_id=model.id,
            display_name=getattr(model, "name", None) or model.id,
            description=getattr(model, "description", None),
            context_length=getattr(model, "context_length", None),
            max_completion_tokens=getattr(
                model,
                "max_completion_tokens",
                None,
            ),
            pricing=pricing,
            supported_features=tuple(
                getattr(model, "supported_features", []) or []
            ),
            supported_sampling_parameters=tuple(
                getattr(model, "supported_sampling_parameters", []) or []
            ),
            input_modalities=tuple(
                getattr(model, "input_modalities", []) or []
            ),
            output_modalities=tuple(
                getattr(model, "output_modalities", []) or []
            ),
        )

In [49]:
normalizer = ModelNormalizer()

profiles = [
    normalizer.normalize(
        provider_id=openai_connection.provider_id,
        model=model,
    )
    for model in models
]

print(f"Normalized models: {len(profiles)}")

Normalized models: 16


In [50]:
for profile in profiles[:3]:
    print(profile)

provider_id='baseten' model_id='openai/gpt-oss-120b' display_name='OpenAI GPT 120B' description='Extremely capable general-purpose LLM with strong, controllable reasoning capabilities' context_length=128072 max_completion_tokens=128072 pricing=ModelPricing(prompt=1e-07, completion=5e-07, input_cache_read=1e-07, image=0.0, request=0.0) supported_features=('tools', 'reasoning', 'json_mode', 'structured_outputs', 'reasoning_effort') supported_sampling_parameters=('temperature', 'stop') input_modalities=('text',) output_modalities=('text',)
provider_id='baseten' model_id='zai-org/GLM-4.7' display_name='GLM 4.7' description='Fast general-purpose LLM with extended 200K context window, superior coding and reasoning capabilities and enhanced tool use for agentic workflows.' context_length=200000 max_completion_tokens=200000 pricing=ModelPricing(prompt=6e-07, completion=2.2e-06, input_cache_read=1.2e-07, image=0.0, request=0.0) supported_features=('tools', 'json_mode', 'structured_outputs') sup

### Normalized Pricing

In [56]:
from decimal import Decimal


class TokenPrice(BaseModel):
    """
    Canonical internal representation of token pricing.

    Internal unit:
        USD per token
    """

    model_config = ConfigDict(frozen=True)

    usd_per_token: Decimal = Field(
        ...,
        ge=0,
    )

    @property
    def usd_per_million_tokens(self) -> Decimal:
        """
        Human-friendly display representation.
        """
        return self.usd_per_token * Decimal("1_000_000")

In [57]:
price = TokenPrice(
    usd_per_token=Decimal("0.0000001")
)

print("Per token:", price.usd_per_token)
print("Per 1M tokens:", price.usd_per_million_tokens)

Per token: 1E-7
Per 1M tokens: 0.1000000


### Model Pricing

In [58]:
class ModelPricing(BaseModel):
    """
    Pricing information normalized into our internal
    USD-per-token representation.
    """

    model_config = ConfigDict(frozen=True)

    prompt: TokenPrice | None = None
    completion: TokenPrice | None = None
    input_cache_read: TokenPrice | None = None
    image: TokenPrice | None = None
    request: TokenPrice | None = None

### Pricing Parser

In [59]:
def parse_token_price(
    value: str | float | int | None,
) -> TokenPrice | None:

    if value is None:
        return None

    return TokenPrice(
        usd_per_token=Decimal(str(value))
    )

In [60]:
price = parse_token_price("0.0000001")

print(price)
print(price.usd_per_million_tokens)

usd_per_token=Decimal('1E-7')
0.1000000


In [66]:
from decimal import Decimal


class TokenPrice(BaseModel):
    """
    Canonical internal representation of token pricing.

    Internal unit:
        USD per token
    """

    model_config = ConfigDict(frozen=True)

    usd_per_token: Decimal = Field(
        ...,
        ge=0,
    )

    @property
    def usd_per_million_tokens(self) -> Decimal:
        """
        Human-friendly display representation.
        """
        return self.usd_per_token * Decimal("1_000_000")


class ModelPricing(BaseModel):
    """
    Pricing information normalized into USD per token.
    """

    model_config = ConfigDict(frozen=True)

    prompt: TokenPrice | None = None
    completion: TokenPrice | None = None
    input_cache_read: TokenPrice | None = None
    image: TokenPrice | None = None
    request: TokenPrice | None = None


class ModelProfile(BaseModel):
    """
    Normalized information discovered from a provider.

    This contains provider facts only.
    Routing intelligence comes later.
    """

    model_config = ConfigDict(frozen=True)

    provider_id: str

    model_id: str
    display_name: str

    description: str | None = None

    context_length: int | None = None
    max_completion_tokens: int | None = None

    pricing: ModelPricing | None = None

    supported_features: tuple[str, ...] = ()
    supported_sampling_parameters: tuple[str, ...] = ()

    input_modalities: tuple[str, ...] = ()
    output_modalities: tuple[str, ...] = ()


def parse_token_price(
    value: str | float | int | None,
) -> TokenPrice | None:

    if value is None:
        return None

    return TokenPrice(
        usd_per_token=Decimal(str(value))
    )

In [67]:
class ModelNormalizer:

    @staticmethod
    def normalize(
        provider_id: str,
        model,
    ) -> ModelProfile:

        pricing_data = getattr(model, "pricing", None) or {}

        pricing = ModelPricing(
            prompt=parse_token_price(
                pricing_data.get("prompt")
            ),

            completion=parse_token_price(
                pricing_data.get("completion")
            ),

            input_cache_read=parse_token_price(
                pricing_data.get("input_cache_read")
            ),

            image=parse_token_price(
                pricing_data.get("image")
            ),

            request=parse_token_price(
                pricing_data.get("request")
            ),
        )

        return ModelProfile(
            provider_id=provider_id,
            model_id=model.id,
            display_name=getattr(model, "name", None) or model.id,
            description=getattr(model, "description", None),
            context_length=getattr(model, "context_length", None),
            max_completion_tokens=getattr(
                model,
                "max_completion_tokens",
                None,
            ),
            pricing=pricing,
            supported_features=tuple(
                getattr(model, "supported_features", []) or []
            ),
            supported_sampling_parameters=tuple(
                getattr(model, "supported_sampling_parameters", []) or []
            ),
            input_modalities=tuple(
                getattr(model, "input_modalities", []) or []
            ),
            output_modalities=tuple(
                getattr(model, "output_modalities", []) or []
            ),
        )

In [68]:
profiles = [
    ModelNormalizer.normalize(
        provider_id=openai_connection.provider_id,
        model=model,
    )
    for model in models
]

print(f"Normalized models: {len(profiles)}")

Normalized models: 16


In [69]:
profile = profiles[0]

print("Model:", profile.model_id)

if profile.pricing:
    if profile.pricing.prompt:
        print(
            "Input:",
            profile.pricing.prompt.usd_per_million_tokens,
            "USD / 1M tokens"
        )

    if profile.pricing.completion:
        print(
            "Output:",
            profile.pricing.completion.usd_per_million_tokens,
            "USD / 1M tokens"
        )

    if profile.pricing.input_cache_read:
        print(
            "Cache read:",
            profile.pricing.input_cache_read.usd_per_million_tokens,
            "USD / 1M tokens"
        )

Model: openai/gpt-oss-120b
Input: 0.1000000 USD / 1M tokens
Output: 0.5000000 USD / 1M tokens
Cache read: 0.1000000 USD / 1M tokens


### Request Capability Requirements

In [70]:
class RequestRequirements(BaseModel):
    """
    Capabilities a model must satisfy for a request.

    These are requirements derived from the request,
    not properties of the model.
    """

    model_config = ConfigDict(frozen=True)

    requires_tools: bool = False

    requires_reasoning: bool = False

    requires_structured_output: bool = False

    requires_vision: bool = False

    minimum_context_tokens: int = Field(
        default=0,
        ge=0,
    )

    minimum_output_tokens: int = Field(
        default=0,
        ge=0,
    )

In [71]:
simple_requirements = RequestRequirements()

print(simple_requirements)

requires_tools=False requires_reasoning=False requires_structured_output=False requires_vision=False minimum_context_tokens=0 minimum_output_tokens=0


In [72]:
complex_requirements = RequestRequirements(
    requires_tools=True,
    requires_reasoning=True,
    requires_structured_output=True,
    requires_vision=True,
    minimum_context_tokens=50_000,
    minimum_output_tokens=8_000,
)

print(complex_requirements)

requires_tools=True requires_reasoning=True requires_structured_output=True requires_vision=True minimum_context_tokens=50000 minimum_output_tokens=8000


### Model Capability Checker

In [77]:
class CapabilityCheckResult(BaseModel):
    """
    Result of checking whether a model can satisfy
    the requirements of a request.
    """

    model_config = ConfigDict(frozen=True)

    model_id: str
    eligible: bool

    reasons: tuple[str, ...] = ()

In [83]:
class ModelCapabilityChecker:

    @staticmethod
    def check(
        model: ModelProfile,
        requirements: RequestRequirements,
    ) -> CapabilityCheckResult:

        reasons = []

        features = set(model.supported_features)
        input_modalities = set(model.input_modalities)

        # Tools
        if requirements.requires_tools:
            if "tools" not in features:
                reasons.append(
                    "tool_calling_not_supported"
                )

        # Reasoning
        if requirements.requires_reasoning:
            if "reasoning" not in features:
                reasons.append(
                    "reasoning_not_supported"
                )

        # Structured outputs
        if requirements.requires_structured_output:
            if "structured_outputs" not in features:
                reasons.append(
                    "structured_outputs_not_supported"
                )

        # Vision
        if requirements.requires_vision:
            if "image" not in input_modalities:
                reasons.append(
                    "vision_not_supported"
                )

        # Context
        if (
            model.context_length is not None
            and model.context_length
            < requirements.minimum_context_tokens
        ):
            reasons.append(
                "insufficient_context_window"
            )

        # Output
        if (
            model.max_completion_tokens is not None
            and model.max_completion_tokens
            < requirements.minimum_output_tokens
        ):
            reasons.append(
                "insufficient_max_output_tokens"
            )

        return CapabilityCheckResult(
            model_id=model.model_id,
            eligible=len(reasons) == 0,
            reasons=tuple(reasons),
        )

In [84]:
checker = ModelCapabilityChecker()

In [85]:
complex_requirements = RequestRequirements(
    requires_tools=True,
    requires_reasoning=True,
    requires_structured_output=True,
    requires_vision=True,
    minimum_context_tokens=50_000,
    minimum_output_tokens=8_000,
)

results = [
    checker.check(
        profile,
        complex_requirements,
    )
    for profile in profiles
]

In [86]:
for result in results:
    if result.eligible:
        print(
            "✅ ELIGIBLE",
            result.model_id,
        )
    else:
        print(
            "❌ REJECTED",
            result.model_id,
            "→",
            ", ".join(result.reasons),
        )

❌ REJECTED openai/gpt-oss-120b → vision_not_supported
❌ REJECTED zai-org/GLM-4.7 → reasoning_not_supported, vision_not_supported
✅ ELIGIBLE moonshotai/Kimi-K2.6
❌ REJECTED deepseek-ai/DeepSeek-V4-Pro → vision_not_supported
❌ REJECTED nvidia/NVIDIA-Nemotron-3-Ultra-550B-A55B → vision_not_supported
✅ ELIGIBLE zai-org/GLM-5.2
✅ ELIGIBLE moonshotai/Kimi-K2.7-Code
❌ REJECTED deepseek-ai/DeepSeek-V4-Flash-0731 → vision_not_supported
✅ ELIGIBLE thinkingmachines/inkling
✅ ELIGIBLE zai-org/GLM-5.2-Fast
✅ ELIGIBLE moonshotai/Kimi-K3
✅ ELIGIBLE thinkingmachines/inkling-small
❌ REJECTED deepseek-ai/DeepSeek-V4-Pro-0813 → vision_not_supported
✅ ELIGIBLE zai-org/GLM-5.3-Flash
❌ REJECTED zai-org/GLM-5.3 → vision_not_supported
✅ ELIGIBLE zai-org/GLM-5.3-Fast


### Cost Estimate

In [88]:
class CostEstimate(BaseModel):
    """
    Estimated inference cost for a single request/model pair.
    """

    model_config = ConfigDict(frozen=True)

    input_tokens: int = Field(..., ge=0)
    output_tokens: int = Field(..., ge=0)

    input_cost_usd: Decimal = Field(..., ge=0)
    output_cost_usd: Decimal = Field(..., ge=0)
    total_cost_usd: Decimal = Field(..., ge=0)

In [89]:
class CostEstimator:

    @staticmethod
    def estimate(
        model: ModelProfile,
        input_tokens: int,
        output_tokens: int,
    ) -> CostEstimate:

        if model.pricing is None:
            raise ValueError(
                f"No pricing information available for "
                f"model '{model.model_id}'"
            )

        if model.pricing.prompt is None:
            raise ValueError(
                f"No input-token pricing available for "
                f"model '{model.model_id}'"
            )

        if model.pricing.completion is None:
            raise ValueError(
                f"No output-token pricing available for "
                f"model '{model.model_id}'"
            )

        input_cost = (
            Decimal(input_tokens)
            * model.pricing.prompt.usd_per_token
        )

        output_cost = (
            Decimal(output_tokens)
            * model.pricing.completion.usd_per_token
        )

        total_cost = input_cost + output_cost

        return CostEstimate(
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            input_cost_usd=input_cost,
            output_cost_usd=output_cost,
            total_cost_usd=total_cost,
        )

In [90]:
# test
cost_estimator = CostEstimator()

profile = profiles[0]

estimate = cost_estimator.estimate(
    model=profile,
    input_tokens=2_000,
    output_tokens=500,
)

print(estimate)

input_tokens=2000 output_tokens=500 input_cost_usd=Decimal('0.0002000') output_cost_usd=Decimal('0.0002500') total_cost_usd=Decimal('0.0004500')


In [91]:
print(
    f"Input cost : ${estimate.input_cost_usd:.8f}"
)

print(
    f"Output cost: ${estimate.output_cost_usd:.8f}"
)

print(
    f"Total cost : ${estimate.total_cost_usd:.8f}"
)

Input cost : $0.00020000
Output cost: $0.00025000
Total cost : $0.00045000


### Model Quality Profile

In [92]:
class ModelQualityProfile(BaseModel):
    """
    Empirically measured quality characteristics of a model.

    These values come from our evaluation benchmark,
    not from provider metadata.
    """

    model_config = ConfigDict(frozen=True)

    model_id: str

    overall_quality: float = Field(
        ...,
        ge=0.0,
        le=1.0,
    )

    reasoning_quality: float = Field(
        ...,
        ge=0.0,
        le=1.0,
    )

    coding_quality: float = Field(
        ...,
        ge=0.0,
        le=1.0,
    )

    extraction_quality: float = Field(
        ...,
        ge=0.0,
        le=1.0,
    )

    factual_quality: float = Field(
        ...,
        ge=0.0,
        le=1.0,
    )

In [93]:
class EvaluationResult(BaseModel):
    """
    Quality measurement for one model answering one request.
    """

    model_config = ConfigDict(frozen=True)

    model_id: str
    request_id: str

    score: float = Field(
        ...,
        ge=0.0,
        le=1.0,
    )

    passed: bool

    evaluator: str

In [94]:
evaluation = EvaluationResult(
    model_id="openai/gpt-oss-120b",
    request_id="req_001",
    score=0.96,
    passed=True,
    evaluator="llm_judge",
)

print(evaluation)

model_id='openai/gpt-oss-120b' request_id='req_001' score=0.96 passed=True evaluator='llm_judge'


### Benchmark Task

In [95]:
from enum import Enum


class TaskType(str, Enum):
    FACTUAL_QA = "factual_qa"
    SUMMARIZATION = "summarization"
    EXTRACTION = "extraction"
    CODING = "coding"
    DEBUGGING = "debugging"
    REASONING = "reasoning"
    LONG_CONTEXT = "long_context"
    VISION = "vision"

In [113]:
class BenchmarkTask(BaseModel):
    """
    A single benchmark request used to compare models.
    """

    model_config = ConfigDict(frozen=True)

    task_id: str
    task_type: TaskType

    prompt: str = Field(..., min_length=1)

    reference_answer: str | None = None

    difficulty: int = Field(..., ge=1, le=5)

    # Used only for cost estimation
    expected_output_tokens: int = Field(..., gt=0)

    # Actual generation ceiling
    max_output_tokens: int = Field(..., gt=0)

    requires_tools: bool = False
    requires_reasoning: bool = False
    requires_structured_output: bool = False
    requires_vision: bool = False

In [140]:
benchmark_tasks = [

    # --------------------------------------------------------
    # 1. Factual QA
    # --------------------------------------------------------

    BenchmarkTask(
        task_id="qa_001",
        task_type=TaskType.FACTUAL_QA,
        prompt="What is the difference between a process and a thread?",
        reference_answer=(
            "A process is an independent program in execution with its own "
            "address space, while threads are execution units within a process "
            "that share the process's memory and resources."
        ),
        difficulty=1,
        expected_output_tokens=150,
        max_output_tokens=1000,
    ),

    # --------------------------------------------------------
    # 2. Summarization
    # --------------------------------------------------------

    BenchmarkTask(
        task_id="sum_001",
        task_type=TaskType.SUMMARIZATION,
        prompt=(
            "Summarize the following text in three concise bullet points:\n\n"
            "Indexes improve database query performance by allowing the database "
            "to locate rows more efficiently instead of scanning the entire table. "
            "They can significantly speed up read-heavy workloads, but they "
            "consume additional storage and can increase the cost of insert, "
            "update, and delete operations."
        ),
        reference_answer=(
            "Indexes speed up data lookups; they reduce unnecessary table scans; "
            "they require additional storage and can add overhead to writes."
        ),
        difficulty=2,
        expected_output_tokens=120,
        max_output_tokens=500,
    ),

    # --------------------------------------------------------
    # 3. Coding
    # --------------------------------------------------------

    BenchmarkTask(
        task_id="code_001",
        task_type=TaskType.CODING,
        prompt=(
            "Write a Python function that returns the length of the longest "
            "consecutive sequence in an unsorted list in O(n) expected time. "
            "Explain the approach and its time and space complexity."
        ),
        reference_answer=None,
        difficulty=3,
        expected_output_tokens=250,
        max_output_tokens=800,
    ),

    # --------------------------------------------------------
    # 4. Debugging
    # --------------------------------------------------------

    BenchmarkTask(
        task_id="debug_001",
        task_type=TaskType.DEBUGGING,
        prompt=(
            "Find the bug in this Python code and explain the fix:\n\n"
            "nums = [1, 2, 3]\n"
            "for i in range(len(nums)):\n"
            "    nums.remove(nums[i])\n\n"
            "Explain why the current implementation behaves incorrectly "
            "and provide a corrected version."
        ),
        reference_answer=None,
        difficulty=3,
        expected_output_tokens=250,
        max_output_tokens=800,
    ),

    # --------------------------------------------------------
    # 5. Multi-step reasoning
    # --------------------------------------------------------

    BenchmarkTask(
        task_id="reason_001",
        task_type=TaskType.REASONING,
        prompt=(
            "A system has three services. A depends on B, B depends on C, "
            "and C depends on A. Explain what architectural problem exists, "
            "why it is harmful, and how you would break the dependency cycle. "
            "Give a concrete example of a better dependency structure."
        ),
        reference_answer=None,
        difficulty=4,
        expected_output_tokens=300,
        max_output_tokens=1000,
        requires_reasoning=True,
    ),

    # --------------------------------------------------------
    # 6. Structured extraction
    # --------------------------------------------------------

    BenchmarkTask(
        task_id="json_001",
        task_type=TaskType.EXTRACTION,
        prompt=(
            "Extract the person's name, age, and city from the following text:\n\n"
            "'Ananya is 24 years old and lives in Chennai.'\n\n"
            "Return the result as JSON."
        ),
        reference_answer='{"name": "Ananya", "age": 24, "city": "Chennai"}',
        difficulty=2,
        expected_output_tokens=100,
        max_output_tokens=300,
        requires_structured_output=True,
    ),
]

In [141]:
print(f"Benchmark size: {len(benchmark_tasks)}")

for task in benchmark_tasks:
    print(
        task.task_id,
        "→",
        task.task_type.value,
        "→ difficulty",
        task.difficulty,
    )

Benchmark size: 6
qa_001 → factual_qa → difficulty 1
sum_001 → summarization → difficulty 2
code_001 → coding → difficulty 3
debug_001 → debugging → difficulty 3
reason_001 → reasoning → difficulty 4
json_001 → extraction → difficulty 2


In [142]:
class EvaluationResult(BaseModel):
    """
    Quality measurement for one model answering one benchmark task.
    """

    model_config = ConfigDict(frozen=True)

    model_id: str
    task_id: str

    score: float = Field(
        ...,
        ge=0.0,
        le=1.0,
    )

    passed: bool

    evaluator: str

    details: str | None = None

In [143]:
from typing import Protocol


class TaskEvaluator(Protocol):

    def evaluate(
        self,
        task: BenchmarkTask,
        response: str,
    ) -> float:
        ...

In [144]:
import json


class ExtractionEvaluator:

    def evaluate(
        self,
        task: BenchmarkTask,
        response: str,
    ) -> float:

        try:
            predicted = json.loads(response)
            expected = json.loads(task.reference_answer)

        except (json.JSONDecodeError, TypeError):
            return 0.0

        if predicted == expected:
            return 1.0

        return 0.0

In [145]:
evaluator = ExtractionEvaluator()

task = next(
    task
    for task in benchmark_tasks
    if task.task_id == "json_001"
)

print(
    evaluator.evaluate(
        task,
        '{"name": "Ananya", "age": 24, "city": "Chennai"}'
    )
)

print(
    evaluator.evaluate(
        task,
        '{"name": "Ananya", "age": 25, "city": "Chennai"}'
    )
)

1.0
0.0


### Model Invocation Result

In [146]:
class ModelResponse(BaseModel):
    """
    Normalized response from any OpenAI-compatible model.
    """

    model_config = ConfigDict(frozen=True)

    request_id: str
    model_id: str

    content: str

    input_tokens: int = Field(
        ...,
        ge=0,
    )

    output_tokens: int = Field(
        ...,
        ge=0,
    )

    total_tokens: int = Field(
        ...,
        ge=0,
    )

In [147]:
class ModelInvoker:

    def __init__(
        self,
        client_factory: OpenAIClientFactory,
    ):
        self.client_factory = client_factory

    def invoke(
        self,
        connection: ProviderConnection,
        model: ModelProfile,
        task: BenchmarkTask,
    ) -> ModelResponse:

        client = self.client_factory.create(
            connection
        )

        response = client.chat.completions.create(
            model=model.model_id,
            messages=[
                {
                    "role": "user",
                    "content": task.prompt,
                }
            ],
            max_tokens=task.max_output_tokens,
        )

        usage = response.usage

        input_tokens = (
            usage.prompt_tokens
            if usage
            else 0
        )

        output_tokens = (
            usage.completion_tokens
            if usage
            else 0
        )

        total_tokens = (
            usage.total_tokens
            if usage
            else input_tokens + output_tokens
        )

        content = response.choices[0].message.content or ""

        return ModelResponse(
            request_id=task.task_id,
            model_id=model.model_id,
            content=content,
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            total_tokens=total_tokens,
        )

In [148]:
model_invoker = ModelInvoker(
    client_factory=client_factory
)

In [149]:
test_model = profiles[0]

print(
    "Using model:",
    test_model.model_id
)

Using model: openai/gpt-oss-120b


In [150]:
task = next(
    task
    for task in benchmark_tasks
    if task.task_id == "qa_001"
)

In [151]:
response = model_invoker.invoke(
    connection=openai_connection,
    model=test_model,
    task=task,
)

In [152]:
print("Model:", response.model_id)
print("Response:")
print(response.content)

print("\nUsage:")
print("Input tokens :", response.input_tokens)
print("Output tokens:", response.output_tokens)
print("Total tokens :", response.total_tokens)

Model: openai/gpt-oss-120b
Response:
**Process vs. Thread – the essential differences**

| Aspect | Process | Thread |
|--------|---------|--------|
| **Definition** | An independent program execution unit that has its own address space, resources (files, sockets, etc.) and at least one thread of control. | The smallest unit of execution that runs inside a process. All threads of a process share the same address space and most resources. |
| **Memory** | Own **virtual memory** (code, data, heap, stack). Isolation: one process cannot directly read/write another’s memory. | **Shared** memory with other threads in the same process (same code segment, global data, heap). Each thread has its own stack and registers. |
| **Creation overhead** | Relatively **heavy** – the OS must allocate a new address space, copy page tables, duplicate file descriptors, etc. | **Lightweight** – the OS only needs to allocate a new stack and register set; most resources are already present. |
| **Context‑switc

### Actual Cost From Real Usage

In [154]:
actual_cost = cost_estimator.estimate(
    model=test_model,
    input_tokens=response.input_tokens,
    output_tokens=response.output_tokens,
)

print("Model:", response.model_id)
print("Input tokens :", actual_cost.input_tokens)
print("Output tokens:", actual_cost.output_tokens)

print(
    f"Input cost   : ${actual_cost.input_cost_usd:.8f}"
)

print(
    f"Output cost  : ${actual_cost.output_cost_usd:.8f}"
)

print(
    f"Total cost   : ${actual_cost.total_cost_usd:.8f}"
)

Model: openai/gpt-oss-120b
Input tokens : 89
Output tokens: 750
Input cost   : $0.00000890
Output cost  : $0.00037500
Total cost   : $0.00038390


In [139]:
task = next(
    task
    for task in benchmark_tasks
    if task.task_id == "qa_001"
)

response = model_invoker.invoke(
    connection=openai_connection,
    model=test_model,
    task=task,
)

print("Model:", response.model_id)
print("Response:")
print(response.content)

print("\nUsage:")
print("Input tokens :", response.input_tokens)
print("Output tokens:", response.output_tokens)
print("Total tokens :", response.total_tokens)

Model: openai/gpt-oss-120b
Response:
**Process vs. Thread – a quick‐look comparison**

| Aspect | Process | Thread |
|--------|---------|--------|
| **Definition** | An independent execution environment that contains its own virtual address space, code, data, and OS resources (open files, handles, etc.). | A lightweight unit of execution that runs inside a process and shares the process’s address space and most resources. |
| **Memory** | Own **private** heap, stack, and global variables. The OS isolates each process’s memory from other processes. | Shares the **same** heap, global variables, and (often) code segment with other threads in the same process. Each thread has its own stack and registers. |
| **Creation cost** | Relatively **expensive** – the OS must allocate a new address space, duplicate (or share) resources, set up a PCB (process control block). | Relatively ** cheap** – the OS only needs to allocate a new thread control block (TCB) and a stack. |
| **Context‑switch over

### Semantic Evaluation Result

In [155]:
class SemanticEvaluation(BaseModel):
    model_config = ConfigDict(frozen=True)

    score: float = Field(
        ...,
        ge=0.0,
        le=1.0,
    )

    passed: bool

    reasoning: str


class SemanticEvaluator:

    def __init__(
        self,
        client_factory: OpenAIClientFactory,
        evaluation_model: str,
        connection: ProviderConnection,
    ):
        self.client_factory = client_factory
        self.evaluation_model = evaluation_model
        self.connection = connection

    def evaluate(
        self,
        task: BenchmarkTask,
        response: str,
        candidate_model_id: str,
    ) -> EvaluationResult:

        client = self.client_factory.create(
            self.connection
        )

        rubric = f"""
You are evaluating the quality of an LLM response.

TASK:
{task.prompt}

REFERENCE ANSWER:
{task.reference_answer or "No exact reference answer is available."}

CANDIDATE RESPONSE:
{response}

Evaluate the candidate on:

1. Factual correctness
2. Completeness
3. Relevance
4. Absence of contradictions

Scoring:

1.00 = Excellent and fully correct
0.95–0.99 = Strong answer with only minor issues
0.90–0.94 = Mostly correct but has meaningful omissions
0.75–0.89 = Partially correct
Below 0.75 = Significant problems

A score >= 0.95 passes.

Return only the structured evaluation.
"""

        evaluation = client.responses.parse(
            model=self.evaluation_model,
            input=rubric,
            text_format=SemanticEvaluation,
        )

        result = evaluation.output_parsed

        return EvaluationResult(
            model_id=candidate_model_id,
            task_id=task.task_id,
            score=result.score,
            passed=result.passed,
            evaluator=self.evaluation_model,
            details=result.reasoning,
        )

In [159]:
evaluation_model = "openai/gpt-oss-120b"

semantic_evaluator = SemanticEvaluator(
    client_factory=client_factory,
    evaluation_model=evaluation_model,
    connection=openai_connection,
)

In [160]:
evaluation = semantic_evaluator.evaluate(
    task=task,
    response=response.content,
    candidate_model_id=response.model_id,
)

print("Model:", evaluation.model_id)
print("Task:", evaluation.task_id)
print("Score:", evaluation.score)
print("Passed:", evaluation.passed)
print("Evaluator:", evaluation.evaluator)
print("Reasoning:", evaluation.details)

Model: openai/gpt-oss-120b
Task: qa_001
Score: 0.99
Passed: True
Evaluator: openai/gpt-oss-120b
Reasoning: The candidate response accurately describes the core distinction between processes and threads as stated in the reference answer, adding correct and relevant details about memory, overhead, context switching, communication, fault isolation, scheduling, use cases, and examples. All information is factually correct, comprehensive, and consistent with the reference. No contradictions or inaccuracies are present, warranting a high score just below perfect due to minor stylistic verbosity but otherwise an excellent answer.
